# LOKLOK qualitative analysis with a local LLM (Colab T4)

**What this notebook does**
1. Cleans the two transcripts (FGD, self-report) and flags noise, repeated fragments and moderator speech.
2. Codes every usable segment with a 25-code codebook using a **local** open model (no text leaves Colab).
3. Verifies every evidence quote against the source text (drops fabricated quotes and counts them).
4. Compares the local model with Claude's first-pass labels, and with **your human labels** if you upload them.
5. Writes tables you can paste into the thesis.

**Before you start**
- Runtime > Change runtime type > **T4 GPU**.
- Upload `loklok_qual_bundle.zip` when asked.
- Do the human coding in `gold_template.csv` **before** you open Claude's labels. That file is your validation set.
- Run the cells in order. After the install cell, restart the session once, then continue from the next cell.


In [ ]:
# 1. GPU check and installs (takes a few minutes). Restart the session after this cell finishes.
!nvidia-smi
!pip -q install vllm rank_bm25 rapidfuzz pyyaml scikit-learn pandas sentence-transformers

In [ ]:
# 2. Settings. Edit only this cell.
MODEL_ID      = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # primary coder (check the repo name on Hugging Face if loading fails)
MODEL_TAG     = "qwen25_7b_awq"                    # used in output file names
RUN_VARIANTS  = [("V0",0), ("V1",0), ("V1",1), ("V2",0)]   # (prompt variant, paraphrase id)
USE_HF_FALLBACK = False   # set True only if vLLM cannot start on your runtime
RUN_DENSE_RETRIEVAL_TEST = True   # bge-m3 retrieval check, runs before the LLM is loaded
MAX_MODEL_LEN = 6144
SEED = 0

In [ ]:
# 3. Unpack the embedded bundle (no upload)
import os, zipfile, sys, base64
from pathlib import Path
BUNDLE_B64 = "..."
zname = '/content/loklok_qual_bundle.zip'
Path(zname).write_bytes(base64.b64decode(BUNDLE_B64))
ROOT = Path('/content/loklok_qual'); ROOT.mkdir(exist_ok=True)
zipfile.ZipFile(zname).extractall(ROOT)
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(sorted(os.listdir(ROOT)))


In [ ]:
# 4. Build units, show data quality, write splits and the human gold template
import json, pandas as pd, numpy as np
import qual_core as q, qual_run as r
from claude_labels_v1 import LABELS, FGD_PARTICIPANT, FGD_UNCLEAR
cb  = q.load_codebook('codebook_v1.yaml'); CODE_IDS = [c['id'] for c in cb['codes']]
units = q.build_units('FGD_transcription.csv', 'selfreport_transcript.json', FGD_PARTICIPANT, FGD_UNCLEAR)
UB = {x.unit_id: x._asdict() for x in units.itertuples()}
ref_rows = q.labels_to_rows(LABELS, units)           # Claude first-pass labels (machine, not human gold)
dev_ids, gold_ids = r.make_splits(units, LABELS)
os.makedirs('out', exist_ok=True)
r.gold_template(units, gold_ids, CODE_IDS, 'out/gold_template.csv')
print(units.groupby(['source','role_ref']).size())
print('credit-artifact segments:', int(units.credit_artifact.sum()), '| dev units:', len(dev_ids), '| units to human-code:', len(gold_ids))

In [ ]:
# 5. (Optional) retrieval check with bge-m3 on GPU, then free the GPU
import torch, gc
if RUN_DENSE_RETRIEVAL_TEST:
    from sentence_transformers import SentenceTransformer
    from rank_bm25 import BM25Okapi
    emb = SentenceTransformer('BAAI/bge-m3', device='cuda'); emb.half()
    ev = units[units.role_ref.isin(q.EVAL_ROLES)].reset_index(drop=True)
    E = emb.encode(list(ev.text_orig), batch_size=16, normalize_embeddings=True, show_progress_bar=False)
    bm = BM25Okapi([t.split() for t in ev.text_norm])
    rel = {c: set(x['unit_id'] for x in ref_rows if any(k['code_id']==c for k in x['codes'])) & set(ev.unit_id) for c in CODE_IDS}
    res = []
    for c in cb['codes']:
        if len(rel[c['id']]) < 3: continue
        qv = emb.encode([c['name'].replace('_',' ') + ': ' + c['definition']], normalize_embeddings=True)[0]
        d_rank = list(ev.unit_id[np.argsort(-(E @ qv))])
        b_rank = list(ev.unit_id[np.argsort(-bm.get_scores(q.norm(c['definition']).split()))])
        rrf = {}
        for rk in (d_rank, b_rank):
            for i, u in enumerate(rk): rrf[u] = rrf.get(u, 0) + 1/(60+i+1)
        h_rank = sorted(rrf, key=lambda u: -rrf[u])
        f = lambda rk: len(set(rk[:10]) & rel[c['id']]) / len(rel[c['id']])
        res.append(dict(code=c['id'], n_rel=len(rel[c['id']]), recall10_dense=f(d_rank), recall10_bm25=f(b_rank), recall10_hybrid=f(h_rank)))
    ret = pd.DataFrame(res); print(ret.round(2).to_string()); print(ret.mean(numeric_only=True).round(2))
    ret.to_csv('out/retrieval_check.csv', index=False)
    del emb; gc.collect(); torch.cuda.empty_cache()
print('free VRAM (GB):', torch.cuda.mem_get_info()[0]/1e9)

In [ ]:
# 6. Load the local LLM (vLLM first, Hugging Face 4-bit as a fallback)
import gc, torch
SCHEMA = q.json_schema(cb)
generate_fn = None
if not USE_HF_FALLBACK:
    from vllm import LLM, SamplingParams
    llm = LLM(model=MODEL_ID, quantization='awq', dtype='half', max_model_len=MAX_MODEL_LEN,
              gpu_memory_utilization=0.85, enforce_eager=True, seed=SEED)
    def make_sp(temp=0.0, seed=SEED):
        kw = dict(temperature=temp, max_tokens=400, seed=seed)
        try:
            from vllm.sampling_params import GuidedDecodingParams
            kw['guided_decoding'] = GuidedDecodingParams(json=SCHEMA)
        except Exception:
            try:
                from vllm.sampling_params import StructuredOutputsParams
                kw['structured_outputs'] = StructuredOutputsParams(json=SCHEMA)
            except Exception:
                pass
        try:
            return SamplingParams(**kw)
        except TypeError:
            kw.pop('guided_decoding', None); kw.pop('structured_outputs', None)
            return SamplingParams(**kw)
    SP = make_sp()
    def generate_fn(batch_msgs):
        outs = llm.chat(batch_msgs, SP, use_tqdm=False)
        return [o.outputs[0].text for o in outs]
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    HF_ID = MODEL_ID.replace('-AWQ', '')
    tok = AutoTokenizer.from_pretrained(HF_ID)
    model = AutoModelForCausalLM.from_pretrained(HF_ID, device_map='auto', torch_dtype=torch.float16,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'))
    def generate_fn(batch_msgs):
        outs = []
        for m in batch_msgs:
            ids = tok.apply_chat_template(m, add_generation_prompt=True, return_tensors='pt').to(model.device)
            g = model.generate(ids, max_new_tokens=350, do_sample=False)
            outs.append(tok.decode(g[0, ids.shape[1]:], skip_special_tokens=True))
        return outs
# quick smoke test on 2 units
t = units[units.unit_id.isin(['F046','S152'])].to_dict('records')
print(generate_fn([q.build_messages(u, cb, 'V1') for u in t]))

In [ ]:
# 7. Run the coding variants (append-only JSONL, safe to re-run after a disconnect)
pool = r.ExemplarPool(units, ref_rows, dev_ids)     # few-shot pool = dev units only, never the evaluation units
import time
RUNS = {}
for v, p in RUN_VARIANTS:
    t0 = time.time()
    path = f'out/{MODEL_TAG}_{v}_p{p}.jsonl'
    RUNS[(v, p)] = r.run_variant(units, cb, generate_fn, v, p, path, pool=pool,
                                 meta=dict(model=MODEL_ID, seed=SEED))
    print(v, p, len(RUNS[(v, p)]), 'units,', round(time.time()-t0), 's')

In [ ]:
# 8. Verify quotes and compare with Claude's labels
summ, tabs = [], {}
for (v, p), rows in RUNS.items():
    kept, tab, s = r.evaluate_run(rows, ref_rows, units, cb, dev_ids, label=f'{MODEL_TAG}_{v}_p{p}')
    grp = r.group_agreement(kept, ref_rows, units, cb, dev_ids)
    tabs[(v, p)] = (kept, tab, grp); summ.append(s)
    tab.to_csv(f'out/agreement_codes_{MODEL_TAG}_{v}_p{p}.csv', index=False)
    grp.to_csv(f'out/agreement_groups_{MODEL_TAG}_{v}_p{p}.csv', index=False)
S = pd.DataFrame(summ)
cols = ['label','assignments','fabrication_rate_exact','parse_failure_rate','insufficient_evidence_rate','micro_f1','polarity_agreement','fgd_role_accuracy_vs_ref']
print(S[cols].round(3).to_string()); S.to_csv(f'out/summary_{MODEL_TAG}.csv', index=False)
best = ('V1', 0)
print('\nGroup-level agreement with Claude (V1, paraphrase 0):'); print(tabs[best][2].round(2).to_string())
print('\nPer-code decisions (pre-specified rule; most codes will be insufficient_to_validate):')
print(tabs[best][1][tabs[best][1].n_ref > 0][['code_id','n_ref','n_test','kappa','kappa_lo','kappa_hi','decision']].round(2).to_string())

In [ ]:
# 9. Stability between the two prompt paraphrases, and closed-vs-context comparison
if ('V1',0) in tabs and ('V1',1) in tabs:
    ub = UB; ids = CODE_IDS
    ev_ids = [u for u in units.unit_id[units.role_ref.isin(q.EVAL_ROLES)] if u not in dev_ids]
    tab, sm = q.agreement_table(tabs[('V1',0)][0], tabs[('V1',1)][0], ub, ids, ev_ids)
    print('Paraphrase stability micro-F1:', round(sm['micro_f1'], 3))

In [ ]:
# 10. Human gold (optional but this is the validation that matters).
# Fill out/gold_template.csv (role + 1 in each code column that applies), upload it as gold_filled.csv.
import os
if os.path.exists('gold_filled.csv') or os.path.exists('/content/gold_filled.csv'):
    gpath = 'gold_filled.csv' if os.path.exists('gold_filled.csv') else '/content/gold_filled.csv'
    gold_rows = r.read_gold(gpath, units, CODE_IDS)
    ub = UB
    ev_ids = [u for u in units.unit_id[units.role_ref.isin(q.EVAL_ROLES)]]
    for name, test in [('Qwen V1', tabs[('V1',0)][0]), ('Claude first pass', ref_rows)]:
        tab, sm = q.agreement_table(gold_rows, test, ub, CODE_IDS, ev_ids)
        tab['decision'] = q.decision(tab)
        print(name, 'micro-F1 vs human:', round(sm['micro_f1'], 3))
        print(tab[tab.n_ref > 0][['code_id','n_ref','n_test','kappa','kappa_lo','kappa_hi','decision']].round(2).to_string())
        tab.to_csv(f"out/human_vs_{name.replace(' ','_')}.csv", index=False)
else:
    print('No gold_filled.csv found. Code out/gold_template.csv first, then upload it and re-run this cell.')

In [ ]:
# 11. Tables for the thesis: prevalence by source and polarity from verified Qwen labels (V1) and from consensus with Claude
kept = tabs[('V1',0)][0]
names = {c['id']: c['name'] for c in cb['codes']}
def flat(rows, tag):
    out = []
    for x in rows:
        if UB[x['unit_id']]['role_ref'] not in q.EVAL_ROLES: continue
        for c in x['codes']: out.append(dict(unit_id=x['unit_id'], source=UB[x['unit_id']]['source'], code_id=c['code_id'], polarity=c['polarity'], who=tag))
    return pd.DataFrame(out)
Fq, Fc = flat(kept, 'qwen'), flat(ref_rows, 'claude')
cons = Fq.merge(Fc, on=['unit_id','code_id'], suffixes=('_q','_c'))
print('assignments  qwen:', len(Fq), ' claude:', len(Fc), ' both:', len(cons))
prev = (cons.groupby(['code_id','source_q']).unit_id.nunique().unstack(fill_value=0)); prev['name'] = prev.index.map(names)
print(prev.sort_values(prev.columns[0], ascending=False).to_string())
prev.to_csv('out/consensus_prevalence.csv'); cons.to_csv('out/consensus_assignments.csv', index=False)

In [ ]:
# 12. Package results for download
import shutil
shutil.make_archive('/content/loklok_qual_results', 'zip', 'out')
try:
    files.download('/content/loklok_qual_results.zip')
except Exception:
    print('Download /content/loklok_qual_results.zip from the file browser.')

## How to read the results (and what not to claim)
- Agreement with Claude measures consistency between two models, not validity. Only agreement with **your** labels (cell 10) is validation.
- With about 190 usable segments, most codes have fewer than 8 positives, so the pre-specified rule marks them *insufficient to validate*. Use the group-level table for the main reliability statement and report per-code results descriptively.
- Counts are segment counts from one researcher-led session and one researcher self-report. They are not prevalence among students.
- Fabrication rate is the share of evidence quotes that are not exact substrings of the source. Report it for every run.